In [1]:
import axelrod as axl
import pandas as pd

In [2]:
p = [player() for player in axl.filtered_strategies({
        "long_run_time": False,
})]


dfs = []

with pd.ExcelWriter(f"All.xlsx") as writer:
    print("Kreiran novi fajl")


for i in range(0, 1):
    print(f"Krećem {i+1}. turnir")
    t = axl.Tournament(players=p, repetitions=1)

    results = t.play()

    results.scores

    
    df = pd.DataFrame(
            {
                "Name": [x.name for x in p],
                "#Wins": list(map(lambda x: sum(x), results.wins)),
                "TotalScore": list(map(lambda x: sum(x), results.scores)),
                "CooperationRating" : results.cooperating_rating,
                "GoodPartnerRating" : results.good_partner_rating,
                "EigenJesusRating" : results.eigenjesus_rating,
                "EigenMosesRating" : results.eigenmoses_rating,
            })


    df = df.sort_values("TotalScore", ascending=False)
    df = df.reset_index(drop=True)
    dfs.append(df)
   

Kreiran novi fajl
Krećem 1. turnir


Analysing: 100%|██████████| 25/25 [00:50<00:00,  2.03s/it]


In [3]:
i = 1
for df in dfs:
    with pd.ExcelWriter(f"All.xlsx", mode="a", engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="Rank_List_"+str(i), startrow=0)
        i = i+1

In [4]:
from functools import reduce

prepared = [
    df[["Name", "TotalScore"]].rename(
        columns={"TotalScore": f"TotalScore{i+1}"}
    )
    for i, df in enumerate(dfs)
]

# Merge svih DF-ova po Name
result_df = reduce(
    lambda left, right: pd.merge(left, right, on="Name", how="outer"),
    prepared
)

with pd.ExcelWriter(f"All.xlsx", mode="a", engine="openpyxl") as writer:
        result_df.to_excel(writer, sheet_name="Total Score", startrow=0)

In [5]:
combined = pd.concat(dfs)
        
agg = combined.groupby('Name').agg({
        '#Wins': 'sum',
        'TotalScore': 'sum',
        'CooperationRating': ['mean', 'std'],
        'GoodPartnerRating': ['mean', 'std'],
        'EigenJesusRating': ['mean', 'std'],
        'EigenMosesRating': ['mean', 'std']
}).reset_index()

agg.columns = ['Name', '#Wins', 'TotalScore',
               'AvgCoopRating', 'SD_CoopRating',
               'AvgGP', 'SD_GP',
               'AvgEJ', 'SD_EJ',
               'AvgEM', 'SD_EM']
        
        # Compute %Wins and AvgScore
num_dfs = len(dfs)
total_matches = (len(p)-1) * num_dfs
agg['%Wins'] = agg['#Wins'] / total_matches
agg['AvgScore'] = agg['TotalScore'] / total_matches
        
        # Keep only desired columns
result = agg[['Name', '%Wins', 'AvgScore',
              'AvgCoopRating', 'SD_CoopRating',
              'AvgGP', 'SD_GP',
              'AvgEJ', 'SD_EJ',
              'AvgEM', 'SD_EM']]
result = result.sort_values(by="AvgScore", ascending=False)


with pd.ExcelWriter(f"All.xlsx", mode="a", engine="openpyxl") as writer:
        result.to_excel(writer, sheet_name="FINAL", startrow=0)

In [6]:
# Kreiramo prazan dict gde će ključevi biti imena, a vrednosti liste rangova
rank_dict = {}

for i, df in enumerate(dfs):
    # Rang po TotalScore u toj iteraciji
    ranks = df.set_index('Name')['TotalScore'].rank(ascending=False, method='min')
    
    for name, rank in ranks.items():
        if name not in rank_dict:
            rank_dict[name] = []
        rank_dict[name].append(rank)

# Pretvaramo u DataFrame
rank_df = pd.DataFrame.from_dict(rank_dict, orient='index')

# Preimenuj kolone u Rank_1, Rank_2, ...
rank_df.columns = [f'Rank_{i+1}' for i in range(len(rank_df.columns))]

# Reset index da Name bude kolona
rank_df = rank_df.reset_index().rename(columns={'index': 'Name'})

# rank_df sada sadrži 30 kolona rangova po iteracijama
with pd.ExcelWriter(f"All_Ranks.xlsx") as writer:
    rank_df.to_excel(writer, startrow=0)